# autoresearch 实验结果分析

[英文原版](analysis.en.ipynb) · [中文学习教程](docs/学习教程.md)

请先在中文版目录 `autoresearch-zh-CN/` 准备真实的 `results.tsv`，并将 Notebook 工作目录设为该目录。本文档不会启动训练。图表会写入 `progress.png`，覆盖同名图片；如需保留仓库自带的示意图，执行前请修改保存文件名。至少需要一条有效基线和一条 `keep` 记录（基线本身可以是 `keep`）。

读取 `results.tsv`，分析自主超参数调整的实验结果。中文图表需要系统已安装中文字体，例如微软雅黑、黑体或 Noto Sans CJK SC；第一个代码单元已配置常用候选字体。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Matplotlib 会按顺序选取系统中可用的字体；中文字体需预先安装
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# 读取 TSV：使用制表符分隔，五列字段名保持不变
df = pd.read_csv("results.tsv", sep="\t")
df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"实验总数： {len(df)}")
print(f"字段： {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("各类实验结果：")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\n保留率： {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# 展示所有保留的实验，即最终留下的改进
kept = df[df["status"] == "KEEP"].copy()
print(f"已保留的实验（共 {len(kept)} 次）：\n")
for i, row in kept.iterrows():
    bpb = row["val_bpb"]
    desc = row["description"]
    print(f"  #{i:3d}  bpb={bpb:.6f}  mem={row['memory_gb']:.1f}GB  {desc}")

## 验证集 BPB 随实验推进的变化

观察已保留实验的最佳 `val_bpb` 如何随实验推进变化。累计最小值表示截至当时找到的最好结果。

原版绘图逻辑会排除崩溃，并重新编号；只突出不高于基线加 0.0005 的点，因此它不是所有失败实验的完整展示。

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# 绘图时排除崩溃记录
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_bpb = valid.loc[0, "val_bpb"]

# 只显示不高于基线加 0.0005 的点，突出关注区域
below = valid[valid["val_bpb"] <= baseline_bpb + 0.0005]

# 用浅灰色背景点表示放弃的实验
disc = below[below["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_bpb"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="放弃")

# 用醒目的绿色点表示保留的实验
kept_v = below[below["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_bpb"],
           c="#2ecc71", s=50, zorder=4, label="保留", edgecolors="black", linewidths=0.5)

# 绘制累计最佳结果的阶梯线
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_bpb = valid.loc[kept_mask, "val_bpb"]
running_min = kept_bpb.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="截至当时的最佳结果")

# 给每个保留的实验标注描述
for idx, bpb in zip(kept_idx, kept_bpb):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, bpb),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("实验序号（排除崩溃后）", fontsize=12)
ax.set_ylabel("验证集 BPB（越低越好）", fontsize=12)
ax.set_title(f"autoresearch 进展：共 {n_total} 次实验，保留 {n_kept} 次", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

# 纵轴范围：略低于最好结果，略高于基线
best_bpb = kept_bpb.min()
margin = (baseline_bpb - best_bpb) * 0.15
ax.set_ylim(best_bpb - margin, baseline_bpb + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("已保存至 progress.png")

## 汇总统计

In [ ]:
# 汇总统计
kept = df[df["status"] == "KEEP"].copy()
baseline_bpb = df.iloc[0]["val_bpb"]
best_bpb = kept["val_bpb"].min()
best_row = kept.loc[kept["val_bpb"].idxmin()]

print(f"基线 val_bpb：  {baseline_bpb:.6f}")
print(f"最佳 val_bpb：      {best_bpb:.6f}")
print(f"总改善幅度： {baseline_bpb - best_bpb:.6f} ({(baseline_bpb - best_bpb) / baseline_bpb * 100:.2f}%)")
print(f"最佳实验：   {best_row['description']}")
print()

# 每次改进对应的实验序号
print("各次改进对应的累计实验进度：")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  实验 #{row['index']:3d}: bpb={row['val_bpb']:.6f}  {desc}")

## 改进排行榜（按保留实验的改善幅度排序）

In [ ]:
# 每次保留实验的改善幅度，相对于上一次保留实验的 BPB 计算
# 实验代码逐步累积：每次都在上一次保留的代码状态上继续修改
kept = df[df["status"] == "KEEP"].copy()
kept["prev_bpb"] = kept["val_bpb"].shift(1)
kept["delta"] = kept["prev_bpb"] - kept["val_bpb"]

# 排除没有前一次对照的基线
hits = kept.iloc[1:].copy()

# 按改善幅度从大到小排序
hits = hits.sort_values("delta", ascending=False)

print(f"{'排名':>4}  {'改善幅度':>8}  {'BPB':>10}  实验描述")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_bpb']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  相对基线的总改善")